In [1]:
# gerar 30 shared base com vários ci nas cláusulas

import json
import random
import numpy as np

candidato_template = {
    "name": "",
    "given": "h < c1 AND vibration >= c2",
    "when": "b > 10 AND dt > 0",
    "do": "action_x",
    "then": "delta_dt >= c3 AND h >= c4"
}

def gerar_c_primos(c, n, std, min_v=0, max_v=100, seed=None):
    if seed is not None:
        random.seed(seed)
    cs = []
    for _ in range(n):
        while True:
            valor = random.normalvariate(c, std)
            # Trunca para o intervalo, mas só aceita se NÃO for exatamente min_v ou max_v
            valor_trunc = max(min(valor, max_v), min_v)
            if valor_trunc != min_v and valor_trunc != max_v:
                cs.append(valor_trunc)
                break
            # Senão, sorteia de novo
    return cs

# Parâmetros principais
n_candidatos = 30

std_dict = {
    "baixo": 5,
    "medio": 15,
    "alto": 30
}

# Inicializa listas vazias para cada nível
candidatos_por_nivel = {
    "baixo": [],
    "medio": [],
    "alto": []
}

# Listas para guardar os desvios padrão dos grupos de cada nível
stds_por_nivel = {
    "baixo": [],
    "medio": [],
    "alto": []
}

for index_grupo in range(100):
    c1 = random.uniform(0.1, 99.9)
    c2 = random.uniform(0.1, 99.9)
    c3 = random.uniform(0.1, 99.9)
    c4 = random.uniform(0.1, 99.9)
    for nivel, std in std_dict.items():
        # Agora geramos um conjunto de ci por candidato
        c1_primos = gerar_c_primos(c1, n_candidatos, std)
        c2_primos = gerar_c_primos(c2, n_candidatos, std)
        c3_primos = gerar_c_primos(c3, n_candidatos, std)
        c4_primos = gerar_c_primos(c4, n_candidatos, std)

        # Usa c1 como referência para calcular o desvio padrão real do grupo
        std_real = np.std(c1_primos)
        stds_por_nivel[nivel].append(std_real)
        print(f"\nNível: {nivel.upper()} - std={std} | Desvio padrão real (c1): {std_real:.2f}")

        candidatos = []
        for i in range(n_candidatos):
            candidato = candidato_template.copy()
            candidato["name"] = f"candidato_{nivel}_{index_grupo+1}_{i+1}"

            c1_str = f"{c1_primos[i]:.0f}"
            c2_str = f"{c2_primos[i]:.0f}"
            c3_str = f"{c3_primos[i]:.0f}"
            c4_str = f"{c4_primos[i]:.0f}"

            # Substitui apenas os marcadores c1, c2, c3, c4
            given = candidato_template["given"].replace("c1", c1_str).replace("c2", c2_str)
            then = candidato_template["then"].replace("c3", c3_str).replace("c4", c4_str)

            candidato["given"] = given
            candidato["then"] = then

            candidatos.append(candidato)

        candidatos_por_nivel[nivel].append(candidatos)

# Mostra a média dos desvios padrão de cada nível (usando c1 como referência)
print("\n" + "="*40)
for nivel in std_dict:
    media_std = np.mean(stds_por_nivel[nivel])
    print(f"MÉDIA dos desvios padrão dos grupos ({nivel.upper()}): {media_std:.2f}")
print("="*40 + "\n")

# Salvar em arquivo JSON
with open("candidatos_por_nivel.json", "w", encoding="utf-8") as f:
    json.dump(candidatos_por_nivel, f, ensure_ascii=False, indent=2)



Nível: BAIXO - std=5 | Desvio padrão real (c1): 3.79

Nível: MEDIO - std=15 | Desvio padrão real (c1): 10.78

Nível: ALTO - std=30 | Desvio padrão real (c1): 12.31

Nível: BAIXO - std=5 | Desvio padrão real (c1): 3.85

Nível: MEDIO - std=15 | Desvio padrão real (c1): 13.67

Nível: ALTO - std=30 | Desvio padrão real (c1): 20.06

Nível: BAIXO - std=5 | Desvio padrão real (c1): 5.71

Nível: MEDIO - std=15 | Desvio padrão real (c1): 12.98

Nível: ALTO - std=30 | Desvio padrão real (c1): 18.66

Nível: BAIXO - std=5 | Desvio padrão real (c1): 3.82

Nível: MEDIO - std=15 | Desvio padrão real (c1): 9.25

Nível: ALTO - std=30 | Desvio padrão real (c1): 19.85

Nível: BAIXO - std=5 | Desvio padrão real (c1): 3.30

Nível: MEDIO - std=15 | Desvio padrão real (c1): 10.63

Nível: ALTO - std=30 | Desvio padrão real (c1): 18.81

Nível: BAIXO - std=5 | Desvio padrão real (c1): 6.67

Nível: MEDIO - std=15 | Desvio padrão real (c1): 12.63

Nível: ALTO - std=30 | Desvio padrão real (c1): 20.78

Nível: BAI

In [2]:
import json

# 1. Ler o arquivo agrupado por nível e grupo
with open("candidatos_por_nivel.json", "r", encoding="utf-8") as f:
    dados = json.load(f)

# 2. Extrair todos os candidatos para uma lista única
todos_candidatos = []
for nivel_lista in dados.values():
    for grupo in nivel_lista:     # grupo é uma lista de candidatos
        todos_candidatos.extend(grupo)

# 3. Salvar como shared_scenarios.json
with open("shared_scenarios.json", "w", encoding="utf-8") as f:
    json.dump(todos_candidatos, f, ensure_ascii=False, indent=2)

print(f"Total de candidatos únicos: {len(todos_candidatos)}")


Total de candidatos únicos: 9000


In [3]:
import pandas as pd
import json
import re

# Substitua pelo nome do seu arquivo jsonl
jsonl_file = "similarity_results.jsonl"

# Lista para guardar as informações extraídas
registros = []

with open(jsonl_file, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        diagnosed_name = data["diagnosed"]["name"]
        candidate_name = data["candidate"]["name"]
        candidate_given = data["candidate"]["given"]

 

        match = re.search(r'h\s*(<=|>=|<|>)\s*([0-9]+(?:\.[0-9]+)?)', candidate_given)
        if match:
            operador, c_linha = match.groups()

        candidate_name

        similarity = data["similarity_result"]
        registros.append({
            "DiagnosedScenario_name": diagnosed_name,
            "CandidateScenario_name": candidate_name,
            "Candidate_given": candidate_given,
            "c_linha": c_linha,
            "similarity": similarity
        })

# Criar DataFrame
df = pd.DataFrame(registros)

split_cols = df["CandidateScenario_name"].str.split('_', expand=True)

# Cria as novas colunas
df['grupo'] = split_cols[1]
df['index_grupo'] = split_cols[2]
df['index_inner_grupo'] = split_cols[3]

df = df[['DiagnosedScenario_name', 'CandidateScenario_name', 
         'grupo', 'index_grupo', 'index_inner_grupo', 
         'Candidate_given', 'c_linha', 'similarity']]


df

,DiagnosedScenario_name,CandidateScenario_name,grupo,index_grupo,index_inner_grupo,Candidate_given,c_linha,similarity
0,diagnosed,candidato_baixo_1_1,baixo,1,1,h < 95 AND vibration >= 94,95,0.584735
1,diagnosed,candidato_baixo_1_2,baixo,1,2,h < 96 AND vibration >= 97,96,0.564057
2,diagnosed,candidato_baixo_1_3,baixo,1,3,h < 95 AND vibration >= 90,95,0.606722
3,diagnosed,candidato_baixo_1_4,baixo,1,4,h < 100 AND vibration >= 95,100,0.570443
4,diagnosed,candidato_baixo_1_5,baixo,1,5,h < 92 AND vibration >= 94,92,0.588535
...,...,...,...,...,...,...,...,...
8995,diagnosed,candidato_alto_100_26,alto,100,26,h < 18 AND vibration >= 54,18,0.668285
8996,diagnosed,candidato_alto_100_27,alto,100,27,h < 21 AND vibration >= 28,21,0.621057
8997,diagnosed,candidato_alto_100_28,alto,100,28,h < 8 AND vibration >= 22,8,0.581135
8998,diagnosed,candidato_alto_100_29,alto,100,29,h < 1 AND vibration >= 2,1,0.529195


In [4]:
idx_max = df.groupby("index_grupo")["similarity"].idxmax()
vencedores = df.loc[idx_max].reset_index(drop=True)
vencedores

,DiagnosedScenario_name,CandidateScenario_name,grupo,index_grupo,index_inner_grupo,Candidate_given,c_linha,similarity
0,diagnosed,candidato_medio_1_7,medio,1,7,h < 76 AND vibration >= 62,76,0.778947
1,diagnosed,candidato_medio_10_10,medio,10,10,h < 68 AND vibration >= 58,68,0.804640
2,diagnosed,candidato_alto_100_13,alto,100,13,h < 74 AND vibration >= 30,74,0.728560
3,diagnosed,candidato_medio_11_1,medio,11,1,h < 43 AND vibration >= 58,43,0.718692
4,diagnosed,candidato_alto_12_27,alto,12,27,h < 69 AND vibration >= 70,69,0.783337
...,...,...,...,...,...,...,...,...
95,diagnosed,candidato_medio_95_19,medio,95,19,h < 66 AND vibration >= 64,66,0.803035
96,diagnosed,candidato_alto_96_2,alto,96,2,h < 67 AND vibration >= 61,67,0.802420
97,diagnosed,candidato_alto_97_10,alto,97,10,h < 66 AND vibration >= 63,66,0.808080
98,diagnosed,candidato_medio_98_6,medio,98,6,h < 75 AND vibration >= 60,75,0.794817


In [5]:
# df[df['index_grupo'] == '1']

In [6]:
contagem_vitorias = vencedores["grupo"].value_counts()
print(contagem_vitorias)

grupo
alto     76
medio    22
baixo     2
Name: count, dtype: int64


In [7]:
# tabela = df.pivot_table(index=['DiagnosedScenario_name', 'index_grupo'], columns='grupo', values='similarity')
# tabela['grupo_vencedor'] = tabela[['baixo', 'medio', 'alto']].idxmax(axis=1)
# tabela


In [8]:
# vitorias = (
#     tabela.groupby('grupo_vencedor')
#       .size()
#       .reset_index(name='qtd_vitorias')
# )
# print(vitorias)


In [9]:
import pandas as pd
import plotly.express as px

# contagem_vitorias -> Series que você já tem
contagem_df = contagem_vitorias.reset_index()
contagem_df.columns = ["grupo", "Contagem"]

map_groups = {"alto": "High", "medio": "Mid", "baixo": "Low"}
contagem_df["grupo_plot"] = contagem_df["grupo"].map(map_groups)

ordem_grupos = ["High", "Mid", "Low"]
contagem_df["grupo_plot"] = pd.Categorical(
    contagem_df["grupo_plot"],
    categories=ordem_grupos,
    ordered=True
)

fig = px.bar(
    contagem_df,
    x="grupo_plot",
    y="Contagem",
    text="Contagem",
    color="grupo_plot",  # ← cada grupo com uma cor
    labels={
        "grupo_plot": "Shared base diversity",
        "Contagem": "#highest-similarity cases",
        "grupo_plot": "Shared base diversity"
    },
    # opcional: definir cores específicas
    color_discrete_map={
        "High": "#1f77b4",   # azul
        "Mid": "#ff7f0e",    # laranja
        "Low": "#2ca02c"     # verde
    }
)

fig.update_traces(textposition="auto")

fig.update_layout(
    width=600,
    height=350,
    margin=dict(l=60, r=20, t=40, b=60),
    showlegend=False  # se quiser esconder a legenda, pode deixar True se quiser mostrar
)

fig.show()


In [10]:
# 1) Tabela bloco × grupo com a MAIOR similaridade de cada grupo em cada bloco
tabela = (
    df
    .groupby(['index_grupo', 'grupo'])['similarity']
    .max()                 # melhor candidato daquele grupo naquele bloco
    .unstack('grupo')      # vira colunas: baixo, medio, alto
)

# Garantir ordem de colunas e remover blocos incompletos (se houver)
tabela = tabela[['baixo', 'medio', 'alto']].dropna()

from scipy.stats import friedmanchisquare

friedman_result = friedmanchisquare(
    tabela['baixo'],
    tabela['medio'],
    tabela['alto']
)

k = 3                    # nº de grupos (baixo, medio, alto)
N = len(tabela)          # nº de blocos (index_grupo válidos)

# Kendall's W = tamanho de efeito do Friedman
kendall_w = friedman_result.statistic / (N * (k - 1))

print(f"Friedman: chi2 = {friedman_result.statistic}, "
      f"df = {k-1}, p-value = {friedman_result.pvalue}")
print(f"Kendall's W (effect size) = {kendall_w}")


Friedman: chi2 = 139.33999999999992, df = 2, p-value = 5.529723878958875e-31
Kendall's W (effect size) = 0.6966999999999995


- Friedman é um teste estatístico não-paramétrico (nao normal) usado para comparar mais de dois grupos pareados.
- Ele verifica se há diferença significativa entre as distribuições dos grupos (a mediana dos ranks dos valores).

- Exemplo clássico:
Imagine que você testa 3 métodos de ensino em 10 alunos e mede a nota de cada aluno em cada método. Como é sempre o “mesmo aluno” sendo comparado, é um dado pareado.

- No seu caso:
Cada linha da tabela é uma comparação para um DiagnosedScenario/index_grupo, e você compara baixo, médio e alto na mesma situação.

In [11]:
# max_sim_df = df.groupby(['DiagnosedScenario_name', 'index_grupo'], as_index=False).agg({
#     'similarity': 'max'
# })

# df_merge = pd.merge(
#     max_sim_df,
#     df[['DiagnosedScenario_name', 'index_grupo', 'grupo', 'similarity']],
#     on=['DiagnosedScenario_name', 'index_grupo', 'similarity'],
#     how='left'
# )

# # Se houver duplicatas (empates), deixa só uma:
# df_merge = df_merge.drop_duplicates(subset=['DiagnosedScenario_name', 'index_grupo'])

# df_merge

In [12]:
# contagem = df_merge['grupo'].value_counts()
# print(contagem)

In [13]:
<><><><><><><><>

SyntaxError: invalid syntax (2122146794.py, line 1)

In [ ]:
# Sempre converta ANTES de filtrar ou agrupar!
df['c_linha'] = pd.to_numeric(df['c_linha'], errors='coerce')
df['similarity'] = pd.to_numeric(df['similarity'], errors='coerce')
df['index_grupo'] = pd.to_numeric(df['index_grupo'], errors='coerce')

temp = df[df["DiagnosedScenario_name"] == "direita_baixa"]
temp = temp[temp["grupo"] == "baixo"]

temp



In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.scatter(temp['c_linha'], temp['similarity'])
plt.xlabel('c_linha (média por grupo)')
plt.ylabel('similarity (média por grupo)')
plt.title('Scatter plot: c_linha vs similarity (agrupado por index_grupo)')
plt.grid(True)
plt.show()

In [ ]:
# Agora pode agrupar
temp = temp.groupby('index_grupo')[['c_linha', 'similarity']].mean().reset_index()
temp = temp.sort_values('similarity', ascending=True)
temp

In [ ]:


import plotly.express as px

fig = px.box(
    temp, 
    y='similarity', 
    points="all", 
    title="Boxplot da Similaridade (tudo junto)"
)
fig.show()


In [ ]:
temp=df[df["DiagnosedScenario_name"] == "direita_baixa"]
temp=temp[temp["grupo"] == "alto"]

temp['c_linha'] = pd.to_numeric(temp['c_linha'], errors='coerce')
temp['similarity'] = pd.to_numeric(temp['similarity'], errors='coerce')
temp['index_grupo'] = pd.to_numeric(temp['index_grupo'], errors='coerce')

temp


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.scatter(temp['c_linha'], temp['similarity'])
plt.xlabel('c_linha (média por grupo)')
plt.ylabel('similarity (média por grupo)')
plt.title('Scatter plot: c_linha vs similarity (agrupado por index_grupo)')
plt.grid(True)
plt.show()

In [ ]:
temp = temp.groupby('index_grupo')[['c_linha', 'similarity']].mean().reset_index()
temp = temp.sort_values('similarity', ascending=True)
temp

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.scatter(temp['c_linha'], temp['similarity'])
plt.xlabel('c_linha (média por grupo)')
plt.ylabel('similarity (média por grupo)')
plt.title('Scatter plot: c_linha vs similarity (agrupado por index_grupo)')
plt.grid(True)
plt.show()

In [ ]:


import plotly.express as px

fig = px.box(
    temp, 
    y='similarity', 
    points="all", 
    title="Boxplot da Similaridade (tudo junto)"
)
fig.show()


In [ ]:

# Exibir as primeiras linhas
print(df.head())

# Se quiser agrupar por DiagnosedScenario_name e CandidateScenario_name:
# (agrupa, mas como são únicos não muda nada, só exemplo)
grouped = df.groupby(["DiagnosedScenario_name", "CandidateScenario_name"]).mean(numeric_only=True)
grouped

In [ ]:
import plotly.express as px

# Filtra primeiro pelo DiagnosedScenario_name
df_direita_baixa = df[df["DiagnosedScenario_name"] == "direita_baixa"]

# Cria uma nova coluna 'tipo' baseada no nome do candidato
def extrair_tipo(nome):
    if "_baixo_" in nome:
        return "baixo"
    elif "_medio_" in nome:
        return "medio"
    elif "_alto_" in nome:
        return "alto"
    else:
        return "outro"

df_direita_baixa["tipo"] = df_direita_baixa["CandidateScenario_name"].apply(extrair_tipo)

# Agora, faz o boxplot com plotly para cada tipo
fig = px.box(df_direita_baixa, x="tipo", y="similarity", points="all", title="Boxplot de Similaridade por Tipo de Candidato")
fig.show()


In [ ]:
import plotly.express as px

# Filtra primeiro pelo DiagnosedScenario_name
df_direita_media = df[df["DiagnosedScenario_name"] == "direita_media"]

# Cria uma nova coluna 'tipo' baseada no nome do candidato
def extrair_tipo(nome):
    if "_baixo_" in nome:
        return "baixo"
    elif "_medio_" in nome:
        return "medio"
    elif "_alto_" in nome:
        return "alto"
    else:
        return "outro"

df_direita_media["tipo"] = df_direita_media["CandidateScenario_name"].apply(extrair_tipo)

# Agora, faz o boxplot com plotly para cada tipo
fig = px.box(df_direita_media, x="tipo", y="similarity", points="all", title="Boxplot de Similaridade por Tipo de Candidato")
fig.show()


In [ ]:
import plotly.express as px

# Filtra primeiro pelo DiagnosedScenario_name
df_direita_alta = df[df["DiagnosedScenario_name"] == "direita_alta"]

# Cria uma nova coluna 'tipo' baseada no nome do candidato
def extrair_tipo(nome):
    if "_baixo_" in nome:
        return "baixo"
    elif "_medio_" in nome:
        return "medio"
    elif "_alto_" in nome:
        return "alto"
    else:
        return "outro"

df_direita_alta["tipo"] = df_direita_alta["CandidateScenario_name"].apply(extrair_tipo)

# Agora, faz o boxplot com plotly para cada tipo
fig = px.box(df_direita_alta, x="tipo", y="similarity", points="all", title="Boxplot de Similaridade por Tipo de Candidato")
fig.show()


In [ ]:
import plotly.express as px


# Cria uma nova coluna 'tipo' baseada no nome do candidato
def extrair_tipo(nome):
    if "_baixo_" in nome:
        return "baixo"
    elif "_medio_" in nome:
        return "medio"
    elif "_alto_" in nome:
        return "alto"
    else:
        return "outro"

df["tipo"] = df["CandidateScenario_name"].apply(extrair_tipo)

# Agora, faz o boxplot com plotly para cada tipo
fig = px.box(df, x="tipo", y="similarity", points="all", title="Boxplot de Similaridade por Tipo de Candidato")
fig.show()
